### Minicurso Sistemas Multi-agente com LangGraph

## Grafo com múltiplos nós e nó decisor

Moacir Antonelli Ponti - 2025

--- 

In [ ]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display


Podemos definir várias funções que representam nós do grafo e conectá-las num fluxo que seja adequado para o problema.

Vamos definir nós para soma e subtração, e um estado que contem a operação executada

In [ ]:
class AgentState(TypedDict):
    values: List[float]
    operation: str
    result: str

def op_soma(state: AgentState) -> AgentState:
    """Soma dos valores no estado e atualiza o resultado."""
    return state


A aresta condicional `add_conditional_edges` permite rotear entre nós

In [ ]:
graph = StateGraph(AgentState)

graph.add_node(node="soma", action=op_soma)


In [ ]:
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
res = app.invoke({"operation":"+", "values":[10,20,30]})
print(res['result'])  # Resultado = 60})

In [ ]:
res = app.invoke({"operation":"-", "values":[100,20,30]})
print(res['result'])

## Exercício: 

Após a primeira operação, realizar uma segunda, que será dividir o valor por 100 se a primeira tiver sido + ou multiplicar por 100 se a primeira tiver sido -

In [ ]:
def decisor_operacao2(state: AgentState) -> str:
    """Decide a segunda operação com base na primeira."""
    if state['operation'] == '+':
        return 'mult'
    elif state['operation'] == '-':
        return 'div'
    
def op_multiply(state: AgentState) -> AgentState:
    """Multiplica o resultado por 100."""
    current_result = int(state['result'].split('=')[1].strip())
    new_result = current_result * 100
    state['result'] = f"Resultado = {new_result}"
    return state

def op_divide(state: AgentState) -> AgentState:
    """Divide o resultado por 100."""
    current_result = int(state['result'].split('=')[1].strip())
    new_result = current_result / 100
    state['result'] = f"Resultado = {new_result}"
    return state

In [ ]:
graph = StateGraph(AgentState)

graph.add_node(node="soma", action=op_soma)
graph.add_node(node="subtracao", action=op_subtracao)
# graph.add_node(node="roteador", action=decisor_operacao) # nao funciona pois nao retorna estado
graph.add_node("roteador1", lambda state:state) # funcao dummy para o roteador
graph.add_node("roteador2", lambda state:state) # funcao dummy para o roteador

graph.add_edge(START, "roteador1")
graph.add_conditional_edges(
    source="roteador1",
    path=decisor_operacao,
    path_map={
        'soma': 'soma',
        'subtracao': 'subtracao'}
)

graph.add_edge("soma", "roteador2")
graph.add_edge("subtracao", "roteador2")

graph.add_node(node="multiplica", action=op_multiply)
graph.add_node(node="divide", action=op_divide)

graph.add_conditional_edges(
    source="roteador2",
    path=decisor_operacao2,
    path_map={"mult":"multiplica",
              "div":"divide"
            },
)    

graph.add_edge("multiplica", END)
graph.add_edge("divide", END)

app = graph.compile()

In [ ]:
from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
res = app.invoke({"operation":"+", "values":[10,20,30]})
print(res['result']) 

In [ ]:
res = app.invoke({"operation":"-", "values":[100,20,30]})
print(res['result']) 